# Model Tester

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np

sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

from tensorflow.keras.layers import Dense, Dropout, Flatten, LeakyReLU, Add
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN, TensorBoard
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts, ExponentialDecay

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import HealpyChebyshev, HealpyPool, Healpy_ResidualLayer

from mlpng import Core
from mlpng.utils import setup_logging, try_init_wandb, make_trainer_plots
from mlpng.utils.dataloaders import MapDataset
from mlpng.utils.callbacks import RMSELoss, RMSELoss2, rmse_metrics

logger = setup_logging("mlpng.trainer", level=logging.DEBUG)

In [3]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print("Python executable:", sys.executable)
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")

Conda environment: ds25
Python executable: /users/stevensonb/.conda/envs/ds25/bin/python
TensorFlow version: 2.15.1
CUDA version: 12.2
cuDNN version: 8


In [4]:
core = Core(
    [
        "settings/n256.json", 
        "--nsims", "100000",     
        "--phi_scale", "333",
        "--shapes", "local",
        "--fnl_range", "-100", "100"
    ]
)

29-Sep-25 15:12:23 - mlpng.core - DEBUG - Parsing CLI args: ['settings/n256.json', '--nsims', '100000', '--phi_scale', '333', '--shapes', 'local', '--fnl_range', '-100', '100']
29-Sep-25 15:12:23 - mlpng.core - INFO - Loading settings from file 'settings/n256.json'
29-Sep-25 15:12:23 - mlpng.core - DEBUG - Forcing setting 'nsims' to 100000 due to CLI
29-Sep-25 15:12:23 - mlpng.core - DEBUG - Forcing setting 'fnl_range' to [-100, 100] due to CLI
29-Sep-25 15:12:23 - mlpng.core - DEBUG - Forcing setting 'phi_scale' to 333.0 due to CLI
29-Sep-25 15:12:23 - mlpng.core - DEBUG - Forcing setting 'shapes' to ['local'] due to CLI
29-Sep-25 15:12:23 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.105e-09
29-Sep-25 15:12:23 - mlpng.core - DEBUG - Overriding cosmo param 'ns' from 0.9624 to 0.965
29-Sep-25 15:12:23 - mlpng.core - DEBUG - Running with settings: 
{
  "cosmo_params": {
    "As": 2.105e-09,
    "ns": 0.965,
    "pivot_scalar": 0.05,
    "H0": 67.4,
    "ombh2": 0

In [5]:
shapes = core.shapes
batch_size = 16
max_epochs = 100
date_time = tf.timestamp().numpy().astype(int)
run_name = f"testing-{date_time}"

print("Run name:", run_name)

Run name: testing-1759176744


In [6]:
ds_split = np.array([0.8, 0.1, 0.1]) * 0.01
n_dups = [25, 10, 2]

ds = MapDataset.fromCore(core, lensed=True)
train, val, test = ds.split(
    train_size=ds_split[0],
    val_size=ds_split[1],
    test_size=ds_split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=n_dups,
    cache_file=f"{core.name}-{run_name}-{core.shapes_str()}",
    gen_batch_size=32,
)

# calculate the number of steps per epoch and decay steps for use later
epoch_steps = math.ceil(core.total_sims * ds_split[0] * n_dups[0] // batch_size)
decay_steps = epoch_steps

print("Steps per epoch:", epoch_steps)

29-Sep-25 15:25:24 - mlpng.utils.dataloaders - DEBUG - Error information from 'data/data/l767_n256_T_100000_p333.0.hdf5'...
29-Sep-25 15:25:24 - mlpng.utils.dataloaders - DEBUG - Fisher Matrix: [0.00869114]
29-Sep-25 15:25:24 - mlpng.utils.dataloaders - DEBUG - Marginal Likelihoods: [10.726587]
29-Sep-25 15:25:24 - mlpng.utils.dataloaders - DEBUG - Fisher for shape local: 0.00834234, std div: 10.94853668315392
29-Sep-25 15:25:24 - mlpng.utils.dataloaders - DEBUG - Splitting 'data/data/l767_n256_T_100000_p333.0.hdf5' into train: 0:800 (800), val: 800:900 (100), test: 900:1000 (100)
29-Sep-25 15:25:24 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/l767_n256_T_100000_p333.0-testing-1759176744-local-train.cache'
29-Sep-25 15:25:24 - mlpng.utils.dataloaders - DEBUG - Using cache file: '/lustre/smuexa01/client/users/stevensonb/tf_cache/l767_n256_T_100000_p333.0-testing-1759176744-local-val.cache'
29-Sep-25 15:25:24 - mlpng.utils.datal

In [7]:
# def normalize_map(x, y):
#     # Calculate mean and std dev across the map pixels (axis=0)
#     # and keep dimensions for broadcasting
#     mean = tf.math.reduce_mean(x, axis=0, keepdims=True)
#     std = tf.math.reduce_std(x, axis=0, keepdims=True)
    
#     # Avoid division by zero for empty or constant patches
#     x = tf.math.divide_no_nan(x - mean, std)
#     return x, y

# # In your data loading cell (VSC-f1311881)
# train = train.map(normalize_map, num_parallel_calls=tf.data.AUTOTUNE)
# val = val.map(normalize_map, num_parallel_calls=tf.data.AUTOTUNE)
# test = test.map(normalize_map, num_parallel_calls=tf.data.AUTOTUNE)

In [8]:
# import matplotlib.pyplot as plt

# # --- Visualize a few samples from the test set ---

# # Get a single batch from the test dataset
# for maps_batch, labels_batch in train.take(1):
#     # Convert tensors to numpy arrays for plotting
#     maps_np = maps_batch.numpy()
#     labels_np = labels_batch.numpy()
    
#     # How many samples from the batch to plot
#     n_samples_to_plot = min(5, len(maps_np))
    
#     print(f"Plotting {n_samples_to_plot} random samples from a test batch...")
    
#     # Create a figure to hold the plots
#     fig = plt.figure(figsize=(12, 3 * n_samples_to_plot))

#     for i in range(n_samples_to_plot):
#         # Get the i-th map and its corresponding label
#         # The map has shape (npix, 1), so we squeeze it to (npix,) for healpy
#         map_to_plot = np.squeeze(maps_np[i])
#         label = labels_np[i][0] # Assuming the first value is the one to display
        
#         # Create a mollweide projection of the map
#         hp.mollview(
#             map_to_plot,
#             sub=(n_samples_to_plot, 1, i + 1),
#             title=f"Test Sample {i+1} (Normalized Map) - fNL = {label:.2f}",
#             unit="Normalized Temperature",
#             nest=True, # The output of alm2map is in RING ordering by default
#         )

#     plt.tight_layout()
#     plt.show()
    
#     # We only need to see one batch, so we break the loop
#     break

In [9]:
class ResidualChebBlock(tf.keras.layers.Layer):
    def __init__(self, K, Fout, activation, dropout_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = HealpyChebyshev(K=K, Fout=Fout, use_bn=True, activation=activation)
        self.conv2 = HealpyChebyshev(K=K, Fout=Fout, use_bn=True, activation=activation)
        self.match = None
        self.dropout = Dropout(dropout_rate) if dropout_rate else None
        self.add = Add()

        self.shortcut_builder = HealpyChebyshev(
            K=1, Fout=Fout, use_bn=True, activation=None
        )

    def build(self, input_shape):
        if input_shape[-1] != self.conv1.Fout:
            self.match = self.shortcut_builder
        super().build(input_shape)

    def call(self, inputs):
        shortcut = self.match(inputs) if self.match else inputs
        x = self.conv1(inputs)
        x = self.conv2(x)
        if self.dropout:
            x = self.dropout(x)
        return self.add([x, shortcut])

def get_model(input_shape, max_batch_size=32, n_out=1, activation=LeakyReLU(0.3)):
    nside = hp.npix2nside(input_shape[1])
    layers = []

    # layers.append(HealpyPool(1, "AVG"))

    n_layers = math.floor(math.log(nside, 2))
    for i in range(n_layers - 1):
        fout = 32 #2 ** (4 + i)
        layers.append(
            Healpy_ResidualLayer(
                "CHEBY",
                {
                    "K": 2, #max(2, 2**(i // 2)),
                    "Fout": fout,
                    "activation": activation,
                    "use_bn": True,
                },
            )
        )
        # layers.append(
        #     HealpyChebyshev(
        #         K=4,
        #         Fout=fout,
        #         # use_bias=True,
        #         use_bn=True,
        #         activation=activation,
        #     )
        # )
        if i < 4:
            # apply dropout only to the first few layers as the data gets too small
            layers.append(Dropout(0.1))
        
        layers.append(HealpyPool(1, "AVG"))

    layers.append(Flatten())
    layers.append(Dropout(0.3))
    layers.append(Dense(64, activation=activation))
    layers.append(Dense(64, activation=activation))
    layers.append(Dropout(0.1))
    layers.append(Dense(32, activation=activation, kernel_initializer='he_normal'))
    layers.append(Dense(32, activation=activation, kernel_initializer='he_normal'))
    layers.append(Dense(n_out))

    model = HealpyGCNN(
        nside,
        indices=np.arange(input_shape[1]),
        layers=layers,
        n_neighbors=8,
        max_batch_size=max_batch_size,
        initial_Fin=input_shape[-1],
    )

    model.build(input_shape)
    return model

In [10]:
strategy = tf.distribute.MirroredStrategy()  # use mirrored strategy for multi-GPU
with strategy.scope():
    learning_rate = 1e-3
    # learning_rate = CosineDecayRestarts(
    #     learning_rate, decay_steps, t_mul=2.0, m_mul=0.9, alpha=0.001
    # )
    learning_rate = ExponentialDecay(
        learning_rate, decay_steps, 0.9, staircase=True
    )

    model = get_model((None, core.npix, core.npols), batch_size, len(shapes))

    model.compile(
        optimizer=AdamW(learning_rate, weight_decay=5e-6),
        loss=RMSELoss(),
        metrics=rmse_metrics(shapes),
    )

In [11]:
model.summary()

Model: "healpy_gcnn_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gcnn__residual_layer (GCNN  (None, 786432, 32)        2240      
 _ResidualLayer)                                                 
                                                                 
 dropout (Dropout)           (None, 786432, 32)        0         
                                                                 
 healpy_pool (HealpyPool)    (None, 196608, 32)        0         
                                                                 
 gcnn__residual_layer_1 (GC  (None, 196608, 32)        4224      
 NN_ResidualLayer)                                               
                                                                 
 dropout_1 (Dropout)         (None, 196608, 32)        0         
                                                                 
 healpy_pool_1 (HealpyPool)  (None, 49152, 32)       

In [12]:
# create the callbacks
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=100, restore_best_weights=True),
]

# if wanted we create a tensorboard and wandb callback, use the CLI to set these
tb_dir = f"{core.dirs['tb']}/{core.name}/{core.slurm.job}"
if core.use_tb:
    callbacks.append(
        TensorBoard(
            log_dir=tb_dir,
            histogram_freq=1,
            write_steps_per_second=True,
        )
    )
    
if core.use_wandb:
    try_init_wandb(
        config={
            "batch_size": batch_size,
            "max_epochs": max_epochs,
            "shapes": shapes,
            "learning_rate": repr(learning_rate),
        },
        dir=core.dirs["wandb"],
        append_to=callbacks,
        patch_tb=core.use_tb,
        patch_logdir=tb_dir,
    )


In [13]:
# get this data that we will need later, also serves to create the test data
# this allows us to have the full cached dataset by the end of the first epoch
# if the cache exists this is very fast
logger.debug("Creating test cache")
y_test = np.concatenate([y for _, y in test])
logger.debug("Test cache created")

29-Sep-25 15:30:56 - mlpng.trainer - DEBUG - Creating test cache
29-Sep-25 15:31:40 - mlpng.trainer - DEBUG - Test cache created


In [14]:
# and finally we fit the model
history = model.fit(
    train,
    epochs=max_epochs,
    validation_data=val,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/100


InvalidArgumentError: Graph execution error:

Detected at node SparseTensorDenseMatMul/SparseTensorDenseMatMul defined at (most recent call last):
  File "/users/stevensonb/.conda/envs/ds25/lib/python3.11/threading.py", line 1002, in _bootstrap

  File "/users/stevensonb/.conda/envs/ds25/lib/python3.11/threading.py", line 1045, in _bootstrap_inner

  File "/users/stevensonb/.conda/envs/ds25/lib/python3.11/site-packages/keras/src/engine/training.py", line 1373, in run_step

  File "/users/stevensonb/.conda/envs/ds25/lib/python3.11/site-packages/keras/src/engine/training.py", line 1150, in train_step

  File "/users/stevensonb/.conda/envs/ds25/lib/python3.11/site-packages/keras/src/utils/traceback_utils.py", line 65, in error_handler

  File "/users/stevensonb/.conda/envs/ds25/lib/python3.11/site-packages/keras/src/engine/training.py", line 590, in __call__

  File "/users/stevensonb/.conda/envs/ds25/lib/python3.11/site-packages/keras/src/utils/traceback_utils.py", line 65, in error_hand

In [ ]:
# model_file = f"{core.dirs['model']}/{core.name}-{run_name}-{core.slurm.job}.keras"
# logger.info("Saving model to %s", model_file)
# model.save(model_file)
# TypeError: Cannot serialize object <deepsphere.healpy_layers.Healpy_ResidualLayer object at 0x15546681ad90> of type <class 'deepsphere.healpy_layers.Healpy_ResidualLayer'>. To be serializable, a class must implement the `get_config()` method.

In [ ]:
# just test loading the model
# with strategy.scope():
#     model = tf.keras.models.load_model(
#         model_file,
#         custom_objects={
#             "HealpyGCNN": HealpyGCNN,
#             "HealpyChebyshev": HealpyChebyshev,
#             "HealpyPool": HealpyPool,
#             "RMSELoss": RMSELoss,
#             # **rmse_metrics(shapes),
#         },
#     )

In [ ]:
logger.debug("Getting final plots")
preds = model.predict(test, verbose=0)
metrics = model.evaluate(test, verbose=0)
plot_dir = os.path.join(core.dirs["plot"], str(core.name), run_name)

make_trainer_plots(
    core,
    plot_dir,
    run_name,
    history,
    metrics,
    y_test,
    preds,
    core.get_likelihoods(True),
    save=False,
    show=True
)

In [ ]:
stop

## OPTuna

Doesnt work with multi-GPU

In [ ]:
import optuna
from optuna.pruners import HyperbandPruner
from optuna.samplers import TPESampler
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.layers import LeakyReLU
from optuna.integration import TFKerasPruningCallback

def get_tuned_model(trial, input_shape, n_out=1):
    """
    A version of get_model modified to accept hyperparameters from an Optuna trial.
    """
    nside = hp.npix2nside(input_shape[1])
    layers = []

    # --- Hyperparameters to Tune ---
    # LeakyReLU alpha
    activation_alpha = trial.suggest_float("activation_alpha", 0.01, 0.4)
    activation = LeakyReLU(activation_alpha)
    
    # Chebyshev filter order
    chebyshev_k = trial.suggest_int("chebyshev_k", 2, 8)
    
    # Initial number of features for convolutional layers
    initial_fout_exp = trial.suggest_int("initial_fout_exp", 3, 5) # 2^3=8 to 2^5=32
    
    # Dropout rate in the convolutional blocks
    conv_dropout = trial.suggest_float("conv_dropout", 0.0, 0.3)

    n_layers = math.floor(math.log(nside, 2))
    for i in range(n_layers):
        fout = 2 ** (initial_fout_exp + i)
        layers.append(
            HealpyChebyshev(K=chebyshev_k, Fout=fout, use_bn=True, activation=activation)
        )
        layers.append(
            HealpyChebyshev(K=chebyshev_k, Fout=fout, use_bn=True, activation=activation)
        )
        # Add dropout as a tunable parameter
        if conv_dropout > 0:
            layers.append(Dropout(conv_dropout))
        
        layers.append(HealpyPool(1, "AVG"))

    layers.append(Flatten())
    
    # Dropout rate for the dense part
    dense_dropout = trial.suggest_float("dense_dropout", 0.2, 0.6)
    layers.append(Dropout(dense_dropout))

    # Number of dense layers and their units
    n_dense_layers = trial.suggest_int("n_dense_layers", 1, 3)
    for i in range(n_dense_layers):
        units = trial.suggest_int(f"dense_units_{i}", 32, 256, log=True)
        layers.append(Dense(units, activation=activation))

    layers.append(Dense(n_out))

    model = HealpyGCNN(
        nside,
        indices=np.arange(input_shape[1]),
        layers=layers,
        n_neighbors=8,
        initial_Fin=input_shape[-1],
    )
    model.build(input_shape)
    return model


def objective(trial):
    """
    The Optuna objective function to be minimized.
    """
    # Clear any previous models from memory
    tf.keras.backend.clear_session()

    # --- Define Hyperparameters for the Optimizer ---
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    
    # --- Create Datasets and Model within the objective ---
    # Note: Using smaller dataset splits for faster tuning. Adjust as needed.
    # ds_split = np.array([0.1, 0.05, 0.05]) # Using 10% of data for faster trials
    # n_dups = [10, 5, 2]

    # ds = MapDataset.fromCore(core, gaussian_mask=True, lensed=True)
    # train_tune, val_tune, _ = ds.split(
    #     train_size=ds_split[0],
    #     val_size=ds_split[1],
    #     test_size=ds_split[2],
    #     to_tf=True,
    #     batch_size=batch_size, # Using the globally defined batch_size
    #     duplicates=n_dups,
    #     gen_batch_size=64,
    # )

    # Build and compile the model with hyperparameters
    strategy = tf.distribute.MirroredStrategy()
    with strategy.scope():
        model = get_tuned_model(trial, (None, core.npix, core.npols), len(shapes))
        model.compile(
            optimizer=AdamW(learning_rate, weight_decay=weight_decay),
            loss=RMSELoss(),
            metrics=rmse_metrics(shapes),
        )

    # --- Callbacks ---
    # Add the TFKerasPruningCallback
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    callbacks = [pruning_callback, TerminateOnNaN()]

    # --- Train the model ---
    history = model.fit(
        train,
        epochs=max_epochs, # Use global max_epochs
        validation_data=val,
        callbacks=callbacks,
        verbose=2, # Set to 0 to keep logs clean
    )

    # Return the value to be minimized (final validation loss)
    return min(history.history["val_loss"])

# --- Run the Optuna Study ---
# 1. Sampler: TPESampler is a form of Bayesian Optimization.
# 2. Pruner: HyperbandPruner stops unpromising trials early.

tuner_dir="data/tuner/"
os.makedirs(tuner_dir, exist_ok=True)

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(),
    pruner=HyperbandPruner(),
    study_name="mlpng_tuning_small",
    storage=f"sqlite:///{tuner_dir}/optuna_study.db"
)

# Start the optimization. `n_trials` is the number of hyperparameter combinations to test.
study.optimize(objective, n_trials=50, show_progress_bar=True)


In [ ]:

# --- Print the results ---
print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")